### Conversie taaksjabloon (word -> json)

In [ ]:
from ask_delphi_api.convert_taaksjabloon import read_dir, convert_doc_to_json
from pathlib import Path
import pprint

project_root = Path.cwd().parent
paths = read_dir(project_root/'Taaksjablonen')

json_coaches = []

for path in paths:
    try:
        print("\n\n")
        json_coach = convert_doc_to_json(path)
        json_coaches.append(json_coach)
    except ValueError as e:
        print(e)

### Naamgeving digicoach

In [ ]:
import json
from ask_delphi_api.import_digicoach import Import
import datetime

json_digicoach = json.loads(json_coaches[0])
naam_digicoach = f"Digicoach {json_digicoach['name']} {datetime.datetime.now()}"
tags= json_digicoach['tags']
name_voorgedefinieerde_zoekopdracht = ""
for tag in tags:
    if tag["type"] == "Keten" or tag["type"] == "Middel":
        name_voorgedefinieerde_zoekopdracht = tag["values"][0]

print(naam_digicoach)
print(name_voorgedefinieerde_zoekopdracht)

### Bronnen uit taaksjabloon registreren als topic

In [ ]:
import_service = Import()

# Ophalen bronnen lijst taaksjabloon
sources = json_digicoach["sources"]

# Aanmaken sources
import_service.create_source_topics(sources)

# Ophalen link list ("Titel","TopicId")
topics = import_service.topic.fetch_topiclist()
import_service.create_link_list(json_digicoach, topics)

### Creatie voorgedefinieerde_zoekopdracht

In [ ]:
from ask_delphi_api.import_digicoach import Import

topic_id_list = []


topic_id_predefined_search, topic_version_id_predefined_search = import_service.create_voorgedefinieerde_zoekopdracht_topic(name_voorgedefinieerde_zoekopdracht)
topic_id_list.append(topic_id_predefined_search)

### Creatie Digicoach

In [ ]:
topic_id_digicoach, topic_version_id_digicoach = import_service.create_digicoach(naam_digicoach, topic_id_predefined_search, topic_version_id_predefined_search)
topic_id_list.append(topic_id_digicoach)

### Tag Digicoach proces

In [ ]:
project_tags = import_service.relation.get_project_tags(topic_id_digicoach, topic_version_id_digicoach)
tags_proces = tags.copy() + [{'type': 'Document_type', 'values': ["Digitale coach"]}]
tags_taak   = tags.copy() + [{'type': 'Document_type', 'values': ["Taak"]}]
tags_stap   = tags.copy() + [{'type': 'Document_type', 'values': ["Stap"]}]

In [ ]:
print("Proces tags aanleggen :")
import_service.relation.add_tags_to_topic(topic_id_digicoach, topic_version_id_digicoach, tags_proces, project_tags)

### Creatie taken, stappen en publiceren

In [ ]:
# Haal takenlijst uit json_digicoach
taken = json_digicoach["tasks"]

# Voor elke taak, maak topic
for taak in taken:

    # Maak topic met naam taak uit json
    topic_id_task, topic_version_id_task = import_service.create_task(taak["name"], topic_id_digicoach, topic_version_id_digicoach)
    topic_id_list.append(topic_id_task)

    # Voeg beschrijving als content toe
    import_service.add_content_to_topic(topic_id_task, topic_version_id_task, taak['description'])

    # Voeg tags aan taak toe
    import_service.relation.add_tags_to_topic(topic_id_task, topic_version_id_task, tags_taak, project_tags)

    # Haal stappenlijst uit taak
    steps = taak["steps"]

    for step in steps:
    # Maak topic met naam step uit json
        topic_id_step, topic_version_id_step = import_service.create_step(step["name"], topic_id_task, topic_version_id_task)
        topic_id_list.append(topic_id_step)

        # Toevoegen beschrijving aan de topic content
        import_service.add_content_to_topic(topic_id_step, topic_version_id_step, step['description'])

        # Aanwezige links toevoegen aan pyramide
        import_service.add_sources(topic_id_task, topic_version_id_task, step['description'], sources)

        # Voeg tags aan stap toe
        import_service.relation.add_tags_to_topic(topic_id_step, topic_version_id_step, tags_stap, project_tags)

        import_service._get_topic().checkin(topic_id_step)

    # Inchecken taak
    import_service._get_topic().checkin(topic_id_task)

import_service._get_topic().checkin(topic_id_digicoach)
import_service._get_topic().checkin(topic_id_predefined_search)

for topic_id in topic_id_list:
    import_service.publiceer(topic_id)
